# Exploratory Data Analysis - Credit Card Fraud Detection

**Objective:** Understand the dataset structure, distributions, and patterns to inform feature engineering and model selection.

**Dataset:** Credit Card Fraud Detection (Kaggle)
- 284,807 transactions over 2 days
- 492 frauds (0.172% of all transactions)
- Features: Time, V1-V28 (PCA components), Amount, Class

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 2. Dataset Overview

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing Values:")
print(df.isnull().sum().sum())
print("\nDuplicate Rows:")
print(df.duplicated().sum())

## 3. Target Variable Analysis

In [ ]:
class_counts = df['Class'].value_counts()
fraud_percentage = (class_counts[1] / len(df)) * 100

print("Class Distribution:")
print(f"Normal Transactions: {class_counts[0]:,} ({100 - fraud_percentage:.3f}%)")
print(f"Fraudulent Transactions: {class_counts[1]:,} ({fraud_percentage:.3f}%)")
print(f"\nImbalance Ratio: {class_counts[0] / class_counts[1]:.2f}:1")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(['Normal', 'Fraud'], class_counts.values, color=['#2ecc71', '#e74c3c'], alpha=0.7)
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Count by Class')
axes[0].set_yscale('log')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')

colors = ['#2ecc71', '#e74c3c']
axes[1].pie(class_counts.values, labels=['Normal', 'Fraud'], autopct='%1.3f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.show()

## 4. Time Analysis

In [ ]:
df['Time_Hour'] = df['Time'] / 3600

print(f"Time Range: {df['Time'].min():.0f}s to {df['Time'].max():.0f}s")
print(f"Duration: {df['Time'].max() / 3600:.1f} hours ({df['Time'].max() / 86400:.1f} days)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].hist(df[df['Class'] == 0]['Time_Hour'], bins=100, alpha=0.7, 
             label='Normal', color='#2ecc71', edgecolor='black')
axes[0].hist(df[df['Class'] == 1]['Time_Hour'], bins=100, alpha=0.7, 
             label='Fraud', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Time (hours)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Transaction Distribution Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

fraud_by_hour = df[df['Class'] == 1].groupby(df['Time_Hour'].astype(int)).size()
axes[1].plot(fraud_by_hour.index, fraud_by_hour.values, marker='o', 
             linewidth=2, markersize=4, color='#e74c3c')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Number of Frauds')
axes[1].set_title('Fraud Transactions Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Amount Analysis

In [ ]:
print("Amount Statistics:")
print(f"\nOverall:")
print(df['Amount'].describe())
print(f"\nNormal Transactions:")
print(df[df['Class'] == 0]['Amount'].describe())
print(f"\nFraudulent Transactions:")
print(df[df['Class'] == 1]['Amount'].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(df[df['Class'] == 0]['Amount'], bins=50, alpha=0.7, 
                label='Normal', color='#2ecc71', edgecolor='black')
axes[0, 0].set_xlabel('Amount')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Amount Distribution - Normal Transactions')
axes[0, 0].set_xlim(0, 1000)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(df[df['Class'] == 1]['Amount'], bins=50, alpha=0.7, 
                label='Fraud', color='#e74c3c', edgecolor='black')
axes[0, 1].set_xlabel('Amount')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Amount Distribution - Fraudulent Transactions')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].boxplot([df[df['Class'] == 0]['Amount'], df[df['Class'] == 1]['Amount']], 
                    labels=['Normal', 'Fraud'], patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
axes[1, 0].set_ylabel('Amount')
axes[1, 0].set_title('Amount Distribution by Class (Boxplot)')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(np.log1p(df[df['Class'] == 0]['Amount']), bins=50, alpha=0.7, 
                label='Normal', color='#2ecc71', edgecolor='black')
axes[1, 1].hist(np.log1p(df[df['Class'] == 1]['Amount']), bins=50, alpha=0.7, 
                label='Fraud', color='#e74c3c', edgecolor='black')
axes[1, 1].set_xlabel('log(Amount + 1)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Log-Transformed Amount Distribution')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. PCA Features Analysis (V1-V28)

In [ ]:
v_features = [f'V{i}' for i in range(1, 29)]

print("PCA Features Statistics:")
print(df[v_features].describe())

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(20, 25))
axes = axes.ravel()

for idx, feature in enumerate(v_features):
    axes[idx].hist(df[df['Class'] == 0][feature], bins=50, alpha=0.6, 
                   label='Normal', color='#2ecc71', density=True)
    axes[idx].hist(df[df['Class'] == 1][feature], bins=50, alpha=0.6, 
                   label='Fraud', color='#e74c3c', density=True)
    axes[idx].set_title(f'{feature} Distribution')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Density')
    axes[idx].legend(loc='upper right', fontsize=8)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
correlation_matrix = df.corr()

plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
class_correlation = df.corrwith(df['Class']).drop('Class').sort_values(ascending=False)

print("Top 10 Features Most Correlated with Fraud:")
print(class_correlation.head(10))
print("\nTop 10 Features Most Negatively Correlated with Fraud:")
print(class_correlation.tail(10))

In [ ]:
plt.figure(figsize=(10, 8))
class_correlation.sort_values().plot(kind='barh', color=['#e74c3c' if x < 0 else '#2ecc71' 
                                                           for x in class_correlation.sort_values()])
plt.xlabel('Correlation with Class (Fraud)')
plt.ylabel('Features')
plt.title('Feature Correlation with Fraud Class')
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Key Insights Summary

In [ ]:
print("=" * 80)
print("KEY FINDINGS FROM EDA")
print("=" * 80)

print("\n1. CLASS IMBALANCE:")
print(f"   - Severe imbalance: {class_counts[0] / class_counts[1]:.2f}:1 ratio")
print(f"   - Only {fraud_percentage:.3f}% of transactions are fraudulent")
print("   - Action: SMOTE, class weights, or ensemble methods needed")

print("\n2. FEATURE ENGINEERING OPPORTUNITIES:")
print("   - Time: No missing values, could create time-based features (hour, day)")
print("   - Amount: Highly skewed, log transformation recommended")
print("   - V features: Already PCA-transformed, mostly normalized")

print("\n3. MOST IMPORTANT FEATURES (based on correlation):")
positive_corr = class_correlation[class_correlation > 0].sort_values(ascending=False).head(3)
negative_corr = class_correlation[class_correlation < 0].sort_values().head(3)
print("   Positive correlation:")
for feat, corr in positive_corr.items():
    print(f"      {feat}: {corr:.4f}")
print("   Negative correlation:")
for feat, corr in negative_corr.items():
    print(f"      {feat}: {corr:.4f}")

print("\n4. DATA QUALITY:")
print(f"   - No missing values")
print(f"   - No duplicate rows")
print(f"   - Clean dataset ready for modeling")

print("\n5. NEXT STEPS:")
print("   - Normalize/scale Amount and Time features")
print("   - Train-test split with stratification")
print("   - Apply SMOTE or other balancing techniques")
print("   - Test multiple algorithms (LR, RF, XGBoost, Isolation Forest)")
print("   - Focus on Precision, Recall, F1, and ROC-AUC metrics")
print("=" * 80)